In [1]:
import pandas as pd
import numpy as np
np.random.seed(0) 

## I Datenprofiling

*Hier wird unseren Datensatz analysiert und bereinigt. Mit dem Ziel auf folgende Fragen zu antworten:*

Struktur: 
- Welche Spalten gibt es und welche Datentypen (Zahlen, Text, Datum) liegen vor?  
Vollständigkeit: 
- Gibt es fehlende Werte?  
Eindeutigkeit:
- Gibt es doppelte Einträge?  
Verteilung & Spannen: 
- Was ist der Mindest-, Höchst- und Durchschnittswert? 
- Gibt es bereits hier extreme Ausreißer?


In [2]:
luxury_datensatz = pd.read_csv("luxury_cosmetics_fraud_analysis_2025.csv")

luxury_datensatz.head() # Es fällt schon auf, dass ein Wert im Spalte Customer_Age in der 4. Zeile fehlt.

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Time,Customer_Age,Customer_Loyalty_Tier,Location,Store_ID,Product_SKU,Product_Category,Purchase_Amount,Payment_Method,Device_Type,IP_Address,Fraud_Flag,Footfall_Count
0,702bdd9b-9c93-41e3-9dbb-a849b2422080,119dca0b-8554-4b2d-9bec-e964eaf6af97,2025-07-27,04:04:15,56.0,Silver,San Francisco,FLAGSHIP-LA,NEBULA-SERUM-07,Concealer,158.24,Mobile Payment,Desktop,239.249.58.237,0,333
1,2e64c346-36bc-4acf-bc2b-8b0fdf46abc5,299df086-26c4-4708-b6d7-fcaeceb14637,2025-03-14,20:23:23,46.0,Platinum,Zurich,BOUTIQUE-SHANGHAI,STELLAR-FOUND-03,Lipstick,86.03,Credit Card,Tablet,84.49.227.90,0,406
2,29ad1278-70ce-421f-8d81-23816b39f4ac,dfa3d24d-b935-49a5-aa1d-7d57a44d8773,2025-02-20,12:36:02,32.0,Silver,Milan,POPUP-TOKYO,SOLAR-BLUSH-04,Mascara,255.69,Gift Card,Desktop,79.207.35.55,0,96
3,07dc4894-e0eb-48f1-99a7-1942b1973d9b,7a67e184-9369-49ee-aeac-18f5b51b230f,2025-04-25,19:09:43,60.0,Bronze,London,BOUTIQUE-NYC,GALAXIA-SET-08,Serum,282.76,Gift Card,Mobile,176.194.167.253,0,186
4,ae407054-5543-429c-918a-cdcc42ea9782,cf14730a-8f5a-453d-b527-39a278852b27,2025-04-17,14:23:23,NaN,Platinum,Miami,BOUTIQUE-NYC,LUNAR-MASC-02,Serum,205.86,Gift Card,Mobile,166.31.46.111,0,179


In [3]:
# Jetzt kugen wir welches Teil diese fehlende Werde für jeden Spalte darstellen.
missing_value_count = luxury_datensatz.isnull().sum()

print(f"Fehlende Werte für jede Spalte \n {missing_value_count}")

#Wie viel Prozent ist das?
total_cells = np.prod(luxury_datensatz.shape)
total_missing = missing_value_count.sum()

# percent of data that is missing
percent_missing = (total_missing/total_cells) * 100
print("-----------------------------------------------------------------------------------------")
print(f"{percent_missing} Prozent des Datensatzes ist fehlend")

Fehlende Werte für jede Spalte 
 Transaction_ID             0
Customer_ID                0
Transaction_Date           0
Transaction_Time           0
Customer_Age             106
Customer_Loyalty_Tier    106
Location                   0
Store_ID                   0
Product_SKU                0
Product_Category           0
Purchase_Amount            0
Payment_Method           106
Device_Type                0
IP_Address                 0
Fraud_Flag                 0
Footfall_Count             0
dtype: int64
-----------------------------------------------------------------------------------------
0.9317862165963432 Prozent des Datensatzes ist fehlend


### Struktur

Wir schauen uns die Datentypen jeder Spalte an, um zu prüfen, ob sie mit dem erwarteten Typ übereinstimmen (z. B. ob `Transaction_Date` wirklich als Datum erkannt wird oder nur als Text/object vorliegt).

In [4]:
luxury_datensatz.info()

<class 'pandas.DataFrame'>
RangeIndex: 2133 entries, 0 to 2132
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Transaction_ID         2133 non-null   str    
 1   Customer_ID            2133 non-null   str    
 2   Transaction_Date       2133 non-null   str    
 3   Transaction_Time       2133 non-null   str    
 4   Customer_Age           2027 non-null   float64
 5   Customer_Loyalty_Tier  2027 non-null   str    
 6   Location               2133 non-null   str    
 7   Store_ID               2133 non-null   str    
 8   Product_SKU            2133 non-null   str    
 9   Product_Category       2133 non-null   str    
 10  Purchase_Amount        2133 non-null   float64
 11  Payment_Method         2027 non-null   str    
 12  Device_Type            2133 non-null   str    
 13  IP_Address             2133 non-null   str    
 14  Fraud_Flag             2133 non-null   int64  
 15  Footfall_Count 

### Eindeutigkeit

Jede Transaktion sollte einen eindeutigen Eintrag darstellen. Wir prüfen deshalb, ob es komplett doppelte Zeilen gibt und ob die `Transaction_ID` (der eigentliche Primärschlüssel) mehrfach vorkommt.

In [5]:
anzahl_duplikate = luxury_datensatz.duplicated().sum()
anzahl_duplikate_id = luxury_datensatz["Transaction_ID"].duplicated().sum()

print(f"Anzahl komplett doppelter Zeilen: {anzahl_duplikate}")
print(f"Anzahl doppelter Transaction_ID: {anzahl_duplikate_id}")

Anzahl komplett doppelter Zeilen: 0
Anzahl doppelter Transaction_ID: 0


### Verteilung & Spannen

Mit `describe()` betrachten wir Minimum, Maximum, Mittelwert und Quartile der numerischen Spalten, um ein Gefühl für die Wertebereiche zu bekommen. Anschließend nutzen wir die IQR-Methode (Interquartilsabstand), um mögliche Ausreißer in `Purchase_Amount`, `Customer_Age` und `Footfall_Count` genauer zu bestimmen.

In [6]:
luxury_datensatz.describe()

,Customer_Age,Purchase_Amount,Fraud_Flag,Footfall_Count
count,2027.000000,2133.000000,2133.000000,2133.000000
mean,41.684262,174.614074,0.030942,272.461791
std,13.718110,72.249043,0.173202,131.113027
min,18.000000,50.260000,0.000000,50.000000
25%,30.000000,113.850000,0.000000,157.000000
50%,42.000000,174.180000,0.000000,269.000000
75%,53.000000,236.360000,0.000000,388.000000
max,65.000000,299.910000,1.000000,500.000000


In [7]:
def iqr_ausreisser(spalte):
    q1 = luxury_datensatz[spalte].quantile(0.25)
    q3 = luxury_datensatz[spalte].quantile(0.75)
    iqr = q3 - q1
    untere_grenze = q1 - 1.5 * iqr
    obere_grenze = q3 + 1.5 * iqr
    ausreisser = luxury_datensatz[(luxury_datensatz[spalte] < untere_grenze) | (luxury_datensatz[spalte] > obere_grenze)]
    print(f"{spalte}: {len(ausreisser)} Ausreißer (Grenzen: {untere_grenze:.2f} bis {obere_grenze:.2f})")

for spalte in ["Purchase_Amount", "Customer_Age", "Footfall_Count"]:
    iqr_ausreisser(spalte)

Purchase_Amount: 0 Ausreißer (Grenzen: -69.92 bis 420.13)
Customer_Age: 0 Ausreißer (Grenzen: -4.50 bis 87.50)
Footfall_Count: 0 Ausreißer (Grenzen: -189.50 bis 734.50)


## II Datenbereinigung

Das Datenprofiling ist abgeschlossen. Jetzt bereinigen wir den Datensatz auf Basis der gefundenen Probleme. Dabei orientiere ich mich an den Schritten des kostenlosen Kaggle-Learn-Kurses ["Data Cleaning"](https://www.kaggle.com/learn/data-cleaning):

- Umgang mit fehlenden Werten
- Parsen von Datumsangaben
- Uneinheitliche Einträge (Inconsistent Data Entry)

### Fehlende Werte behandeln

Wie im Kaggle-Kurs empfohlen, schauen wir uns zuerst genauer an, *wo* die fehlenden Werte auftreten, bevor wir entscheiden, wie wir mit ihnen umgehen: Treten sie in denselben Zeilen auf (mögliches System-Muster) oder verteilt über den ganzen Datensatz (eher zufällig)?

In [8]:
betroffene_spalten = ["Customer_Age", "Customer_Loyalty_Tier", "Payment_Method"]

anzahl_fehlender_werte_pro_zeile = luxury_datensatz[betroffene_spalten].isnull().sum(axis=1).value_counts()
print(f"Anzahl fehlender Werte (von 3 betroffenen Spalten) pro Zeile: \n{anzahl_fehlender_werte_pro_zeile}")

Anzahl fehlender Werte (von 3 betroffenen Spalten) pro Zeile: 
0    1830
1     288
2      15
Name: count, dtype: int64


Die fehlenden Werte überschneiden sich nur teilweise: 288 Zeilen fehlt genau ein Wert, 15 Zeilen fehlen zwei Werte, und in keiner Zeile fehlen alle drei Werte gleichzeitig. Das spricht eher für unabhängig zufällig verteilte fehlende Werte je Spalte als für ein systematisches Muster (z. B. ein einzelner fehlerhafter Datenimport). Deshalb imputieren wir jede Spalte unabhängig, statt Zeilen zu löschen:

- `Customer_Age` (numerisch) → Median, da robust gegenüber Ausreißern.
- `Customer_Loyalty_Tier` und `Payment_Method` (kategorisch) → Platzhalter `"Unbekannt"`, damit die fehlende Angabe sichtbar bleibt, statt künstlich einer bestehenden Kategorie (z. B. per Modus) zugeordnet zu werden.

In [9]:
luxury_datensatz["Customer_Age"] = luxury_datensatz["Customer_Age"].fillna(luxury_datensatz["Customer_Age"].median())
luxury_datensatz["Customer_Loyalty_Tier"] = luxury_datensatz["Customer_Loyalty_Tier"].fillna("Unbekannt")
luxury_datensatz["Payment_Method"] = luxury_datensatz["Payment_Method"].fillna("Unbekannt")

print(luxury_datensatz.isnull().sum())

Transaction_ID           0
Customer_ID              0
Transaction_Date         0
Transaction_Time         0
Customer_Age             0
Customer_Loyalty_Tier    0
Location                 0
Store_ID                 0
Product_SKU              0
Product_Category         0
Purchase_Amount          0
Payment_Method           0
Device_Type              0
IP_Address               0
Fraud_Flag               0
Footfall_Count           0
dtype: int64


### Datumsangaben parsen

`Transaction_Date` liegt laut der `info()`-Ausgabe oben noch als Text vor. Nach dem Kaggle-Kurs-Kapitel "Parsing Dates" prüfen wir zuerst, ob alle Werte demselben Format entsprechen, und wandeln die Spalte anschließend mit `pd.to_datetime()` in ein echtes Datumsformat um, damit spätere zeitbasierte Auswertungen möglich sind.

In [10]:
laengen = luxury_datensatz["Transaction_Date"].str.len().value_counts()
print(f"Länge der Datum-Strings: \n{laengen}\n")

luxury_datensatz["Transaction_Date"] = pd.to_datetime(luxury_datensatz["Transaction_Date"], format="%Y-%m-%d")

print(f"Neuer Datentyp: {luxury_datensatz['Transaction_Date'].dtype}")
print(f"Zeitraum: {luxury_datensatz['Transaction_Date'].min().date()} bis {luxury_datensatz['Transaction_Date'].max().date()}")

Länge der Datum-Strings: 
Transaction_Date
10    2133
Name: count, dtype: int64

Neuer Datentyp: datetime64[us]
Zeitraum: 2025-02-14 bis 2025-08-12


### Uneinheitliche Einträge (Inconsistent Data Entry)

Wie im Kaggle-Kurs beschrieben, prüfen wir die kategorischen Spalten auf uneinheitliche Schreibweisen (z. B. Groß-/Kleinschreibung oder überflüssige Leerzeichen), die eigentlich dieselbe Kategorie meinen würden. Dazu vergleichen wir die Anzahl eindeutiger Werte vor und nach dem Trimmen/Kleinschreiben je Spalte.

In [11]:
kategorische_spalten = ["Customer_Loyalty_Tier", "Location", "Product_Category", "Payment_Method", "Device_Type"]

for spalte in kategorische_spalten:
    vorher = luxury_datensatz[spalte].nunique()
    nachher = luxury_datensatz[spalte].str.strip().str.lower().nunique()
    print(f"{spalte}: {vorher} eindeutige Werte vorher, {nachher} nach Trimmen/Kleinschreiben")

Customer_Loyalty_Tier: 6 eindeutige Werte vorher, 6 nach Trimmen/Kleinschreiben
Location: 20 eindeutige Werte vorher, 20 nach Trimmen/Kleinschreiben
Product_Category: 10 eindeutige Werte vorher, 10 nach Trimmen/Kleinschreiben
Payment_Method: 5 eindeutige Werte vorher, 5 nach Trimmen/Kleinschreiben
Device_Type: 4 eindeutige Werte vorher, 4 nach Trimmen/Kleinschreiben


Die Anzahl eindeutiger Werte ändert sich in keiner Spalte durch Trimmen/Kleinschreiben — es liegen also keine uneinheitlichen Einträge (z. B. `"gold"` vs. `"Gold "`) vor.

### Zwischenfazit

- **Struktur:** 16 Spalten, überwiegend Text (`str`), zwei numerische Spalten (`float64`) und zwei Ganzzahl-Spalten (`int64`); `Transaction_Date` wurde von Text in `datetime64` umgewandelt.
- **Vollständigkeit:** `Customer_Age`, `Customer_Loyalty_Tier` und `Payment_Method` hatten je 106 fehlende Werte (~5 % der Zeilen), überwiegend unabhängig voneinander verteilt. Nach der Imputation ist der Datensatz vollständig.
- **Eindeutigkeit:** Keine doppelten Zeilen und keine doppelten `Transaction_ID` — jede Transaktion ist eindeutig.
- **Verteilung & Spannen:** Alle numerischen Spalten liegen in plausiblen Wertebereichen (z. B. `Customer_Age` 18–65, `Purchase_Amount` 50,26–299,91 €); laut IQR-Methode gibt es keine extremen Ausreißer.
- **Datumsformat:** Alle `Transaction_Date`-Werte folgen demselben Format (`YYYY-MM-DD`), Zeitraum 2025-02-14 bis 2025-08-12.
- **Uneinheitliche Einträge:** In den kategorischen Spalten wurden keine inkonsistenten Schreibweisen gefunden.

Damit ist das Datenprofiling und die darauf aufbauende Datenbereinigung abgeschlossen. Als Nächstes folgen die automatisierten Quality-Checks.